## Import

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import torch
import matplotlib.pyplot as plt

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

%load_ext autoreload
%autoreload 2

## Dataset

In [ ]:
from datasets.MoT import MoT

n, d = 10000, 16                                 # num of samples; data dimensionality
c = 2                                            # num of mixture components

dataset = MoT(n_samples=n, n_dims=d, n_components=c, seed=42)

Z = dataset.sample_data(n_samples=n).to(device)
H = dataset.entropy()

print('data size', Z.size())
print('entropy', H)
print('weights', dataset.weights)
print('centers', dataset.centers)
print('half_widths', dataset.half_widths)

## 1D visualization

In [ ]:
# Visualize the 1D mixture PDF with more components
dataset_1d = MoT(n_samples=5000, n_dims=1, n_components=10, seed=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: PDF
dataset_1d.plot_pdf_1d(ax=axes[0])

# Right: histogram of samples vs PDF
samples_1d = dataset_1d.sample_data(10000).numpy()
axes[1].hist(samples_1d, bins=50, density=True, alpha=0.5, label='Samples')
from datasets.MoT import _triangle_pdf
x_min = (dataset_1d.centers - dataset_1d.half_widths).min() - 0.5
x_max = (dataset_1d.centers + dataset_1d.half_widths).max() + 0.5
x_grid = np.linspace(x_min, x_max, 1000)
total_pdf = sum(
    dataset_1d.weights[k] * _triangle_pdf(x_grid, dataset_1d.centers[k], dataset_1d.half_widths[k])
    for k in range(dataset_1d.n_components)
)
axes[1].plot(x_grid, total_pdf, 'k-', linewidth=2, label='True PDF')
axes[1].set_title('Samples vs True PDF')
axes[1].legend()

plt.tight_layout()
plt.show()

## Copula estimate

In [ ]:
from GC import GC

gc = GC()
gc.learn(Z)
log_probs = gc.log_probs(Z)

H_gc = -log_probs.mean().item()

print('H', H)
print('H_gc', H_gc)

## Visualizing marginals

In [ ]:
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

for j in range(2):
    ax = axes[j]
    J = j * d // 2

    x = Z[:, J].cpu().numpy()
    kde = gc.marginals[J]
    x_grid = np.linspace(np.min(x) - 1, np.max(x) + 1, 1000)
    kde_vals = kde.evaluate(x_grid)

    ax.hist(x, bins=30, density=True, alpha=0.5, label='Histogram')
    ax.plot(x_grid, kde_vals, label='KDE', color='black')
    ax.set_title(f'Dimension {J + 1}')
    ax.legend()

plt.tight_layout()
plt.show()